In [25]:
import os
import pandas as pd

df_path = "."
data = {}

for file in os.listdir(df_path):
    if file.endswith('.csv'):
        df_name = file.replace('olist_','').replace('_dataset.csv','').replace('.csv','')
        path = os.path.join(df_path,file)
        data[df_name] = pd.read_csv(path)

print(f"loaded")


loaded


In [26]:
print(data.keys())

dict_keys(['customers', 'geolocation', 'orders', 'order_items', 'order_payments', 'order_reviews', 'products', 'sellers', 'product_category_name_translation'])


In [27]:
print(f"total tables: {len(data)}")

total tables: 9


In [28]:
data['customers'].info()
data['geolocation'].info()
data['orders'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  

In [35]:
data['orders'].describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,late_delivery
count,99441,99281,97658,96476,99441,96476.000000
mean,2017-12-31 08:43:12.776581120,2017-12-31 18:35:24.098800128,2018-01-04 21:49:48.138278656,2018-01-14 12:09:19.035542272,2018-01-24 03:08:37.730111232,-11.876881
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00,-147.000000
25%,2017-09-12 14:46:19,2017-09-12 23:24:16,2017-09-15 22:28:50.249999872,2017-09-25 22:07:22.249999872,2017-10-03 00:00:00,-17.000000
50%,2018-01-18 23:04:36,2018-01-19 11:36:13,2018-01-24 16:10:58,2018-02-02 19:28:10.500000,2018-02-15 00:00:00,-12.000000
75%,2018-05-04 15:42:16,2018-05-04 20:35:10,2018-05-08 13:37:45,2018-05-15 22:48:52.249999872,2018-05-25 00:00:00,-7.000000
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00,188.000000
std,NaN,NaN,NaN,NaN,NaN,10.183854


DATA CLEANING

In [29]:
for name, df in data.items():
    null_counts = df.isnull().sum()
    missing_columns = null_counts[null_counts > 0]

    if not missing_columns.empty:
        print(f"Table '{name}' has missing values :")
        print(missing_columns, "\n")

Table 'orders' has missing values :
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64 

Table 'order_reviews' has missing values :
review_comment_title      87656
review_comment_message    58247
dtype: int64 

Table 'products' has missing values :
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64 



In [32]:
# cleaning 'order_reviews' table 
data['order_reviews']['review_comment_title'] = data['order_reviews']['review_comment_title'].fillna('No Title')
data['order_reviews']['review_comment_message'] = data['order_reviews']['review_comment_message'].fillna('No Message')

# cleaning 'products' table
data['products']['product_category_name'] = data['products']['product_category_name'].fillna('Unknown')
data['products'].dropna(subset=['product_weight_g'], inplace=True)

# cleaning 'orders' table
timestamp_columns = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in timestamp_columns:
    data['orders'][col] = pd.to_datetime(data['orders'][col])

print("Data cleaning completed")



Data cleaning completed


In [33]:
data['orders']['late_delivery'] = (
    data['orders']['order_delivered_customer_date'] - data['orders']['order_estimated_delivery_date']
).dt.days
print(data['orders'][['order_id', 'late_delivery']].head())

                           order_id  late_delivery
0  e481f51cbdc54678b7cc49136f2d6af7           -8.0
1  53cdb2fc8bc7dce0b6741e2150273451           -6.0
2  47770eb9100c2d0c44946d9cf07ec65d          -18.0
3  949d5b44dbf5de918fe9c16f97b45f8a          -13.0
4  ad21c59c0840e6cb83a9ceb5573f8159          -10.0


In [ ]:
from sqlalchemy import create_engine

username = "root"
password = "Your@Password"
host = "localhost"
port = "3306"
database = "olist_db"


engine= create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")


for table_name , df in data.items():
    print(f" table: '{table_name}'...")
    df.to_sql(
        name=table_name,
        con= engine, 
        if_exists="replace",
        index=False)





 table: 'customers'...
 table: 'geolocation'...
 table: 'orders'...
 table: 'order_items'...
 table: 'order_payments'...
 table: 'order_reviews'...
 table: 'products'...
 table: 'sellers'...
 table: 'product_category_name_translation'...
